# 07 · Spark SQL

Everything you've done with the DataFrame API you can also do in plain **SQL** — and you can mix the two
freely. Both compile to the *same* optimized plan, so there's no performance difference. Use whichever
reads better for the task (many teams write transformations in SQL).

In [1]:
# Setup — run this cell first every time you open the notebook
import os, sys
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")   # Spark runs on Java
os.environ["PYSPARK_PYTHON"] = sys.executable                         # workers use this same Python

from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = (SparkSession.builder
         .appName("pyspark-course")
         .master("local[*]")                               # run locally, use all CPU cores
         .config("spark.sql.shuffle.partitions", "8")      # explained in lesson 10
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                   # keep Java's log noise out of the notebook

DATA, OUT = "../data", "../output"

In [2]:
orders = spark.read.csv(f"{DATA}/orders.csv", header=True, inferSchema=True)
products = spark.read.csv(f"{DATA}/products.csv", header=True, inferSchema=True)
customers = spark.read.csv(f"{DATA}/customers.csv", header=True, inferSchema=True)

# register DataFrames as temporary views so SQL can see them
orders.createOrReplaceTempView("orders")
products.createOrReplaceTempView("products")
customers.createOrReplaceTempView("customers")

In [3]:
spark.sql("SELECT status, COUNT(*) AS n FROM orders GROUP BY status ORDER BY n DESC").show()

+---------+----+
|   status|   n|
+---------+----+
|completed|4035|
| returned| 503|
|cancelled| 462|
+---------+----+



Multi-line SQL with joins and a CTE (`WITH`):

In [4]:
result = spark.sql("""
    WITH completed AS (
        SELECT o.*, p.category, o.quantity * p.price AS revenue
        FROM orders o
        JOIN products p ON o.product_id = p.product_id
        WHERE o.status = 'completed'
    )
    SELECT c.country, ROUND(SUM(revenue), 2) AS revenue, COUNT(*) AS n_orders
    FROM completed
    JOIN customers c USING (customer_id)
    GROUP BY c.country
    ORDER BY revenue DESC
""")
result.show()

+-------+---------+--------+
|country|  revenue|n_orders|
+-------+---------+--------+
| Brazil|229596.94|     682|
|  Japan|218443.82|     623|
|     US|201436.42|     586|
| France| 189150.0|     551|
|     UK|174995.81|     470|
|Germany|159386.24|     453|
|  Spain|124768.88|     363|
|   NULL|103417.94|     307|
+-------+---------+--------+



`spark.sql` returns an ordinary DataFrame, so you can keep going with the Python API:

In [5]:
result.filter(F.col("country").isNotNull()).withColumn("avg_order", F.round(F.col("revenue") / F.col("n_orders"), 2)).show(3)

+-------+---------+--------+---------+
|country|  revenue|n_orders|avg_order|
+-------+---------+--------+---------+
| Brazil|229596.94|     682|   336.65|
|  Japan|218443.82|     623|   350.63|
|     US|201436.42|     586|   343.75|
+-------+---------+--------+---------+
only showing top 3 rows


### Same plan either way

In [6]:
spark.sql("SELECT customer_id, COUNT(*) AS n FROM orders WHERE quantity > 2 GROUP BY customer_id").explain()
orders.filter(F.col("quantity") > 2).groupBy("customer_id").agg(F.count("*").alias("n")).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[customer_id#18], functions=[count(1)])
   +- Exchange hashpartitioning(customer_id#18, 8), ENSURE_REQUIREMENTS, [plan_id=638]
      +- HashAggregate(keys=[customer_id#18], functions=[partial_count(1)])
         +- Project [customer_id#18]
            +- Filter (isnotnull(quantity#20) AND (quantity#20 > 2))
               +- FileScan csv [customer_id#18,quantity#20] Batched: false, DataFilters: [isnotnull(quantity#20), (quantity#20 > 2)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/path/to/pyspark-tutorial/data/orders.csv], PartitionFilters: [], PushedFilters: [IsNotNull(quantity), GreaterThan(quantity,2)], ReadSchema: struct<customer_id:int,quantity:int>


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[customer_id#18], functions=[count(1)])
   +- Exchange hashpartitioning(customer_id#18, 8), ENSURE_REQUIREMENTS, [plan_id=657]
      +- HashAggregate(keys=[customer_i

## SQL snippets inside the DataFrame API
* `F.expr("...")` — any SQL expression as a Column
* `df.selectExpr("...", "...")` — select with SQL expressions
* `df.filter("...")` — filter with a SQL condition

In [7]:
orders.selectExpr("order_id", "quantity * 2 AS double_qty", "upper(status) AS status").show(3)
orders.withColumn("is_big", F.expr("CASE WHEN quantity >= 4 THEN true ELSE false END")).show(3)
orders.filter("status = 'returned' AND quantity > 3").show(3)

+--------+----------+---------+
|order_id|double_qty|   status|
+--------+----------+---------+
|       1|         4|COMPLETED|
|       2|         2|COMPLETED|
|       3|        10|COMPLETED|
+--------+----------+---------+
only showing top 3 rows
+--------+-----------+----------+--------+-------------------+---------+------+
|order_id|customer_id|product_id|quantity|           order_ts|   status|is_big|
+--------+-----------+----------+--------+-------------------+---------+------+
|       1|         26|        27|       2|2024-09-11 08:18:52|completed| false|
|       2|          9|        23|       1|2024-04-18 15:22:01|completed| false|
|       3|         93|         4|       5|2024-04-01 16:43:45|completed|  true|
+--------+-----------+----------+--------+-------------------+---------+------+
only showing top 3 rows
+--------+-----------+----------+--------+-------------------+--------+
|order_id|customer_id|product_id|quantity|           order_ts|  status|
+--------+-----------+--

## Parameterised queries
Never paste user input into SQL with f-strings (SQL injection!). Use named parameters:

In [8]:
spark.sql("SELECT * FROM orders WHERE status = :status AND quantity >= :q",
          args={"status": "returned", "q": 5}).show(3)

+--------+-----------+----------+--------+-------------------+--------+
|order_id|customer_id|product_id|quantity|           order_ts|  status|
+--------+-----------+----------+--------+-------------------+--------+
|      84|        180|        27|       5|2024-09-17 05:56:15|returned|
|     176|        108|         8|       5|2024-08-26 23:44:00|returned|
|     219|         36|         1|       5|2024-09-21 13:10:10|returned|
+--------+-----------+----------+--------+-------------------+--------+
only showing top 3 rows


## Views and the catalog
* `createOrReplaceTempView` — visible to this SparkSession only, disappears when it stops.
* `createOrReplaceGlobalTempView` — shared across sessions in the same app, lives in the `global_temp` database.
* `df.write.saveAsTable("name")` — a permanent table in a metastore (on Databricks this is how you create real tables).

In [9]:
for t in spark.catalog.listTables():
    print(t.name, t.tableType)

customers TEMPORARY
orders TEMPORARY
products TEMPORARY


### Exercise 1
Using **`spark.sql`**, create `sql_rev_by_cat`: completed-order revenue per category, columns **`category`, `revenue`** (rounded to 2 dp).

In [10]:
sql_rev_by_cat = spark.sql("""
    SELECT p.category, ROUND(SUM(o.quantity * p.price), 2) AS revenue
    FROM orders o JOIN products p ON o.product_id = p.product_id
    WHERE o.status = 'completed'
    GROUP BY p.category
""")

In [11]:
# Check — run me after your answer
got = {r.category: r.revenue for r in sql_rev_by_cat.collect()}
exp = {'Books': 346914.81, 'Clothing': 174227.66, 'Electronics': 270553.15, 'Home': 234515.31, 'Toys': 374985.12}
assert got.keys() == exp.keys() and all(abs(got[k] - exp[k]) < 0.05 for k in exp), got
print("Correct!")

Correct!


### Exercise 2
Using **SQL with a window function** (`ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)`), create `sql_top2`: the **top 2 products by completed revenue in each category**. Columns **`category`, `product_name`, `revenue`, `rnk`**.

In [12]:
sql_top2 = spark.sql("""
    WITH per_product AS (
        SELECT p.category, p.product_name, SUM(o.quantity * p.price) AS revenue
        FROM orders o JOIN products p ON o.product_id = p.product_id
        WHERE o.status = 'completed'
        GROUP BY p.category, p.product_name
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY category ORDER BY revenue DESC) AS rnk
        FROM per_product
    )
    SELECT category, product_name, revenue, rnk FROM ranked WHERE rnk <= 2
""")

In [13]:
# Check — run me after your answer
assert sql_top2.columns == ["category", "product_name", "revenue", "rnk"]
assert sorted(r.product_name for r in sql_top2.collect()) == ['Cookbook', 'Jacket', 'Jeans', 'Lamp', 'Monitor', 'Pillow', 'Plush Bear', 'Poetry Book', 'Puzzle', 'Speaker']
print("Correct!")

Correct!


### Exercise 3
Using **`selectExpr`** on `orders`, create `sx` with columns **`order_id`** and **`qty_x10`** (quantity × 10).

In [14]:
sx = orders.selectExpr("order_id", "quantity * 10 AS qty_x10")

In [15]:
# Check — run me after your answer
assert sx.columns == ["order_id", "qty_x10"]
assert sx.agg(F.sum("qty_x10")).first()[0] == orders.agg(F.sum("quantity")).first()[0] * 10
print("Correct!")

Correct!
